# MU/MUU 历史日 K 研究

本笔记本只做本地历史数据研究，不连接下单接口。

## tl;dr

- 完整样本外区间为 2025-10-14 至 2026-07-16，共 189 个交易日；周频风险再平衡策略回报 17.33%，最大回撤 3.06%。
- 相同周频风险仓位基准回报 17.33%，与均线策略完全一致；这说明三个样本外窗口中均线过滤没有提供增量。
- 不受风险上限约束的一次买入持有回报 48.25%，但最大回撤 30.76%、最差单日 -14.61%，风险显著更高。
- MUU 对 MU 的日收益 beta 为 1.990，日收益相关系数 0.9981；长期累计仍明显偏离理想化的每日 2 倍复利路径。
- 结论仅用于下一轮策略设计，不构成盈利保证或实盘批准。

## Context & Methods

### Key Assumptions

- MU 只负责产生信号，执行始终使用已批准的 MUU 替代规则。
- 只允许整股、不开融资、不做空；信号在收盘生成，下一交易日开盘成交。
- 初始资金为 1,500 美元，风险暴露目标为账户权益的 10%；MUU 按 2 倍暴露折算，因此名义持仓目标约为权益的 5%。
- 采用锚定式 walk-forward：252 日训练、63 日测试，只保留完整测试窗口；预设周频（5 个交易日）再平衡。
- 手续费和滑点来自 `configs/paper.toml`。

In [1]:
from pathlib import Path
from IPython.display import Markdown, display
import json
import sys

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from us_quant.config import load_config
from us_quant.research import run_daily_k_research

research = run_daily_k_research(
    load_config(ROOT / 'configs' / 'paper.toml'),
    data_root=ROOT / 'data',
)
research['scope']

{'bar_size': '1 day',
 'price_type': 'IBKR historical TRADES, regular session',
 'signal_symbol': 'MU',
 'execution_symbol': 'MUU',
 'execution_policy': 'always use approved MUU substitute',
 'whole_shares_only': True,
 'margin_borrowing': False,
 'initial_equity': 1500.0,
 'target_risk_weight': 0.1,
 'first_date': '2024-10-10',
 'last_date': '2026-07-23',
 'aligned_rows': 446}

## Data

先检查日期、重复值、OHLC 合法性和 MUU 对 MU 的日收益跟踪。

In [2]:
display(Markdown('### 日 K 数据质量'))
display(research['data_quality']['profiles'])
display(Markdown('### MUU 跟踪特征'))
display(research['data_quality']['tracking'])
display(Markdown('### 数据质量说明'))
display(research['data_quality']['issues'])

### 日 K 数据质量

{'MU': {'row_count': 1254,
  'first_date': '2021-07-26',
  'last_date': '2026-07-23',
  'sorted_dates': True,
  'duplicate_dates': 0,
  'invalid_ohlc_rows': 0,
  'zero_volume_rows': 0,
  'max_daily_return': 0.19291611185086552,
  'min_daily_return': -0.16179018286814245,
  'daily_volatility': 0.03482335295110676},
 'MUU': {'row_count': 446,
  'first_date': '2024-10-10',
  'last_date': '2026-07-23',
  'sorted_dates': True,
  'duplicate_dates': 0,
  'invalid_ohlc_rows': 0,
  'zero_volume_rows': 0,
  'max_daily_return': 0.3847305389221557,
  'min_daily_return': -0.3252032520325203,
  'daily_volatility': 0.08989313555469222}}

### MUU 跟踪特征

{'observation_count': 445,
 'daily_return_correlation': 0.9981226686845167,
 'daily_beta_to_mu': 1.9895138468159361,
 'annualized_tracking_error_vs_2x_mu': 0.08772120387651247,
 'mu_cumulative_return': 8.3690036900369,
 'muu_cumulative_return': 26.02962962962963,
 'daily_reset_2x_mu_cumulative_return': 34.2268838224895,
 'muu_minus_daily_reset_2x_mu': -8.19725419285987}

### 数据质量说明

[{'severity': 'medium',
  'code': 'short_muu_history',
  'message': 'The aligned MU/MUU history is under two full trading years, so regime coverage is limited.'}]

## Results

所有百分比结果均来自执行后账户权益曲线，已经计入配置中的佣金和滑点。

In [3]:
def pct(value):
    return f'{value:.2%}'

rows = []
for label, key in (
    ('均线 + 周频风险再平衡', 'strategy'),
    ('周频恒定风险基准', 'weekly_constant_risk_benchmark'),
    ('一次买入持有（风险漂移）', 'buy_and_hold_benchmark'),
):
    metrics = research['out_of_sample'][key]
    rows.append({
        '方案': label,
        '回报': pct(metrics['total_return']),
        '最大回撤': pct(metrics['max_drawdown']),
        '最差单日': pct(metrics['worst_day']),
        '佣金': f"${metrics['commission']:.2f}",
        '成交次数': metrics['trade_count'],
    })
rows

[{'方案': '均线 + 周频风险再平衡',
  '回报': '17.33%',
  '最大回撤': '3.06%',
  '最差单日': '-0.98%',
  '佣金': '$11.55',
  '成交次数': 33},
 {'方案': '周频恒定风险基准',
  '回报': '17.33%',
  '最大回撤': '3.06%',
  '最差单日': '-0.98%',
  '佣金': '$11.55',
  '成交次数': 33},
 {'方案': '一次买入持有（风险漂移）',
  '回报': '48.25%',
  '最大回撤': '30.76%',
  '最差单日': '-14.61%',
  '佣金': '$0.35',
  '成交次数': 1}]

In [4]:
display(Markdown('### Walk-forward 分段'))
display(research['out_of_sample']['folds'])
display(Markdown('### 调仓频率敏感性'))
display(research['frequency_sensitivity'])
display(Markdown('### 成本压力测试'))
display(research['cost_stress'])
display(Markdown('### 资金规模敏感性'))
display(research['capital_sensitivity'])

### Walk-forward 分段

[{'fold': 1,
  'train_start': '2024-10-10',
  'train_end': '2025-10-13',
  'test_start': '2025-10-14',
  'test_end': '2026-01-13',
  'selected_parameters': '20/100',
  'training_sharpe': 2.1448803667924374,
  'training_return': 0.05616256,
  'training_max_drawdown': 0.01071955707044559,
  'oos_return': 0.05576366666666667,
  'oos_commission': 4.55,
  'oos_trade_count': 13},
 {'fold': 2,
  'train_start': '2024-10-10',
  'train_end': '2026-01-13',
  'test_start': '2026-01-14',
  'test_end': '2026-04-15',
  'selected_parameters': '20/100',
  'training_sharpe': 2.285054648872754,
  'training_return': 0.11187762933333334,
  'training_max_drawdown': 0.022228180712465052,
  'oos_return': 0.03762787694594529,
  'oos_commission': 3.85,
  'oos_trade_count': 11},
 {'fold': 3,
  'train_start': '2024-10-10',
  'train_end': '2026-04-15',
  'test_start': '2026-04-16',
  'test_end': '2026-07-16',
  'selected_parameters': '20/100',
  'training_sharpe': 2.1416795643037356,
  'training_return': 0.1502653

### 调仓频率敏感性

[{'frequency': 'daily',
  'rebalance_interval_days': 1,
  'oos_return': 0.13937513733333334,
  'no_cost_oos_return': 0.16743333333333332,
  'cost_drag': 0.02805819599999998,
  'max_drawdown': 0.03333156941025808,
  'commission': 31.85,
  'trade_count': 91,
  'selected_parameters': ['20/100', '20/100', '20/100']},
 {'frequency': 'weekly',
  'rebalance_interval_days': 5,
  'oos_return': 0.173284244,
  'no_cost_oos_return': 0.18108666666666667,
  'cost_drag': 0.00780242266666667,
  'max_drawdown': 0.030608803896615577,
  'commission': 11.55,
  'trade_count': 33,
  'selected_parameters': ['20/100', '20/100', '20/100']},
 {'frequency': 'monthly',
  'rebalance_interval_days': 21,
  'oos_return': 0.19337168,
  'no_cost_oos_return': 0.19506,
  'cost_drag': 0.001688320000000021,
  'max_drawdown': 0.033275287907915375,
  'commission': 2.45,
  'trade_count': 7,
  'selected_parameters': ['10/50', '20/100', '20/100']}]

### 成本压力测试

[{'scenario': 'no_cost',
  'slippage_bps': 0.0,
  'minimum_commission': 0.0,
  'oos_return': 0.18108666666666667,
  'max_drawdown': 0.030260827072261926,
  'commission': 0.0,
  'trade_count': 33,
  'selected_parameters': ['20/100', '20/100', '20/100']},
 {'scenario': 'configured',
  'slippage_bps': 2.0,
  'minimum_commission': 0.35,
  'oos_return': 0.173284244,
  'max_drawdown': 0.030608803896615577,
  'commission': 11.55,
  'trade_count': 33,
  'selected_parameters': ['20/100', '20/100', '20/100']},
 {'scenario': 'slippage_5bps',
  'slippage_bps': 5.0,
  'minimum_commission': 0.35,
  'oos_return': 0.17313061,
  'max_drawdown': 0.030613783734235518,
  'commission': 11.55,
  'trade_count': 33,
  'selected_parameters': ['20/100', '20/100', '20/100']},
 {'scenario': 'slippage_10bps',
  'slippage_bps': 10.0,
  'minimum_commission': 0.35,
  'oos_return': 0.17287455333333332,
  'max_drawdown': 0.030622084748834866,
  'commission': 11.55,
  'trade_count': 33,
  'selected_parameters': ['20/100

### 资金规模敏感性

[{'initial_equity': 1000.0,
  'oos_return': 0.156543998,
  'max_drawdown': 0.029006598169394625,
  'commission': 9.45,
  'trade_count': 27,
  'selected_parameters': ['20/100', '20/100', '20/100']},
 {'initial_equity': 1500.0,
  'oos_return': 0.173284244,
  'max_drawdown': 0.030608803896615577,
  'commission': 11.55,
  'trade_count': 33,
  'selected_parameters': ['20/100', '20/100', '20/100']},
 {'initial_equity': 2000.0,
  'oos_return': 0.172899502,
  'max_drawdown': 0.03323306238463315,
  'commission': 11.9,
  'trade_count': 34,
  'selected_parameters': ['20/100', '20/100', '20/100']}]

## Takeaways

1. **先不批准模拟盘自动下单。** 当前样本外收益为正，但只有三个完整窗口，且处于 MU/MUU 强趋势阶段。
2. **周频优于日频主要来自更少的整股来回调整。** 这证明小资金账户不能简单把频率拉满；最低佣金会显著侵蚀结果。
3. **均线过滤尚未证明有效。** 它在样本外阶段与恒定风险基准完全一致，下一轮要增加不同市场状态或更长的替代数据历史，而不是继续微调参数追求更高历史收益。
4. **一次买入持有不是当前风险方案。** 它在强趋势中赚得更多，但风险暴露随价格漂移，最差单日和最大回撤明显放大。
5. **日 K 无法研究日内做 T。** 日内策略必须等分钟级数据，并单独模拟价差、成交概率和当日往返成本。